In [ ]:
import requests
import json
import os

def get_cik(ticker):
    """
    Retrieves the CIK for a given ticker symbol from the SEC's AutilPI.
    """
    url = f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={ticker}&type=&dateb=&owner=exclude&start=0&count=1&output=atom"
    headers = {'User-Agent': "wildandzaky4@gmail.com"}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        # Extract CIK from the response
        try:
            cik = response.text.split('<cik>')[1].split('</cik>')[0]
            return cik.zfill(10)  # Pad with leading zeros to ensure 10 digits
        except IndexError:
            print("Could not parse CIK from SEC response.")
            return None
    else:
        print(f"Failed to retrieve data for ticker {ticker}. Status code: {response.status_code}")
        return None

# Main part of the script
ticker = "TSLA"  # Example ticker, replace with desired ticker
cik = get_cik(ticker)

if cik:
    print(f"CIK for {ticker}: {cik}")
    # Construct the URL
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

    # Make the request
    headers = {'User-Agent': "wildandzaky4@gmail.com"}  # Required by SEC
    response = requests.get(url, headers=headers)

    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()

        # Create directory if it doesn't exist
        output_dir = "/root/vynixmodelling/dataset/sec_data"
        os.makedirs(output_dir, exist_ok=True)

        # Save the JSON data to a file
        filename = os.path.join(output_dir, f"{ticker}.json")
        with open(filename, 'w') as f:
            json.dump(data, f, indent=4)  # indent for pretty printing

        print(f"Data saved to {filename}")
    else:
        print(f"Failed to retrieve data. Status code: {response.status_code}")
        print(response.text)  # Print the response text for debugging
else:
    print(f"Could not retrieve CIK for ticker {ticker}.")

CIK for TSLA: 0001318605
Data saved to /root/vynixmodelling/dataset/sec_data/TSLA.json


In [ ]:
import json
import pandas as pd
import os
from datetime import datetime

# Load the JSON file
with open('/root/vynixmodelling/dataset/sec_data/TSLA.json', 'r') as file:
    data = json.load(file)

# Get ticker symbol from the file
ticker = "TSLA"

# Create directory if it doesn't exist
os.makedirs('/root/vynixmodelling/dataset/data_fundamental', exist_ok=True)

# Function to extract quarterly data for a given metric
def extract_quarterly_data(metric_name):
    quarterly_data = []
    
    if metric_name in data["facts"]["us-gaap"] and "units" in data["facts"]["us-gaap"][metric_name]:
        metric_data = data["facts"]["us-gaap"][metric_name]
        
        # Most financial data is in USD
        if "USD" in metric_data["units"]:
            # Track used period keys to avoid duplicates
            used_periods = set()
            
            # Sort entries by filed date (latest first) to get most recent values
            sorted_entries = sorted(
                metric_data["units"]["USD"],
                key=lambda x: x.get("filed", "0000-00-00"),
                reverse=True
            )
            
            for entry in sorted_entries:
                # Check if fp exists, is not None, and starts with Q or is FY with form 10-K
                if ("fp" in entry and 
                    entry["fp"] is not None and 
                    isinstance(entry["fp"], str) and 
                    (entry["fp"].startswith("Q") or 
                     (entry["fp"] == "FY" and entry.get("form") == "10-K")) and 
                    "fy" in entry and 
                    "val" in entry):
                    
                    # Treat FY as Q4 if form is 10-K
                    period_fp = "Q4" if entry["fp"] == "FY" else entry["fp"]
                    period_key = f"{entry['fy']}-{period_fp}"
                    
                    # Skip if we already have this period for this metric
                    if period_key in used_periods:
                        continue
                        
                    used_periods.add(period_key)
                    
                    quarterly_data.append({
                        "date": period_key,
                        "value": entry["val"],
                        "metric": metric_name,
                        "filed_date": entry.get("filed", "")
                    })
    
    return quarterly_data

# Get all available metrics
all_available_metrics = list(data["facts"]["us-gaap"].keys())
print(f"Found {len(all_available_metrics)} total metrics in the data")

# Collect all quarterly data
all_quarterly_data = []
metrics_with_data = 0

# Process each metric and collect data
for i, metric in enumerate(all_available_metrics):
    metric_data = extract_quarterly_data(metric)
    
    if metric_data:
        all_quarterly_data.extend(metric_data)
        metrics_with_data += 1
    
    # Print progress every 100 metrics
    if (i + 1) % 100 == 0 or i == len(all_available_metrics) - 1:
        print(f"Processed {i+1}/{len(all_available_metrics)} metrics")

print(f"Found data for {metrics_with_data} metrics out of {len(all_available_metrics)} total metrics")

# Create DataFrame
if all_quarterly_data:
    df = pd.DataFrame(all_quarterly_data)
    
    # Verify no duplicates in date-metric combinations
    duplicate_check = df.duplicated(subset=['date', 'metric'])
    if duplicate_check.any():
        print(f"Warning: Found {duplicate_check.sum()} duplicate entries. Keeping only the first occurrence.")
        df = df.drop_duplicates(subset=['date', 'metric'])
    
    # Pivot the data to get metrics as columns
    pivoted_df = df.pivot(index='date', columns='metric', values='value')
    
    # Sort by date (assuming format YYYY-QX)
    pivoted_df = pivoted_df.sort_index()
    
    # Create filename with ticker
    filename = f"/root/vynixmodelling/dataset/data_fundamental/{ticker}_time.csv"
    
    # Save to CSV
    pivoted_df.to_csv(filename)
    print(f"Data saved to {filename}")
    print(f"Saved {len(pivoted_df)} quarters of data for {len(pivoted_df.columns)} metrics")
    
    # Show statistics
    print(f"\nDataFrame statistics:")
    print(f"Number of quarters: {len(pivoted_df)}")
    print(f"Number of metrics: {len(pivoted_df.columns)}")
    print(f"Number of data points: {pivoted_df.count().sum()}")
    print(f"Data completeness: {(pivoted_df.count().sum() / (len(pivoted_df) * len(pivoted_df.columns)) * 100):.2f}%")
    
    # Display the first few rows
    print("\nSample of saved data (first 5 rows, first 5 columns):")
    print(pivoted_df.iloc[:5, :5])
else:
    print("No quarterly data found for any metrics")

Found 616 total metrics in the data
Processed 100/616 metrics
Processed 200/616 metrics
Processed 300/616 metrics
Processed 400/616 metrics
Processed 500/616 metrics
Processed 600/616 metrics
Processed 616/616 metrics
Found data for 538 metrics out of 616 total metrics
Data saved to /root/vynixmodelling/dataset/data_fundamental/TSLA_time.csv
Saved 57 quarters of data for 538 metrics

DataFrame statistics:
Number of quarters: 57
Number of metrics: 538
Number of data points: 8631
Data completeness: 28.15%

Sample of saved data (first 5 rows, first 5 columns):
metric   AccountsAndNotesReceivableNet  AccountsPayableCurrent  \
date                                                             
2011-Q2                            NaN              28951000.0   
2011-Q3                            NaN              28951000.0   
2011-Q4                            NaN              28951000.0   
2012-Q1                            NaN              56141000.0   
2012-Q2                            NaN  

In [18]:
from fundamental_feature_engineering import *

print('85 Feature Engineering Functions telah didefinisikan!')


85 Feature Engineering Functions telah didefinisikan!


In [19]:
# Membuat dataframe baru dengan filtered_df dan 85 fungsi tambahan

# Pertama, identifikasi kolom yang memiliki data lengkap
complete_columns = []
threshold = 0.7  # Minimal 70% data tersedia

for col in pivoted_df.columns:
    completeness = pivoted_df[col].count() / len(pivoted_df)
    if completeness >= threshold:
        complete_columns.append(col)

print(f'Kolom dengan data lengkap (>= {threshold*100}%): {len(complete_columns)}')
print(f'Kolom: {complete_columns[:10]}...')  # Tampilkan 10 pertama

# Buat filtered_df dengan kolom yang memiliki data lengkap
filtered_df = pivoted_df[complete_columns].copy()

# Fill missing values dengan forward fill dan backward fill
filtered_df = filtered_df.fillna(method='ffill').fillna(method='bfill')

print(f'\nFiltered DataFrame shape: {filtered_df.shape}')
print(f'Data completeness setelah filtering: {(filtered_df.count().sum() / (len(filtered_df) * len(filtered_df.columns)) * 100):.2f}%')

# Terapkan 85 fungsi feature engineering
enhanced_df = filtered_df.copy()

# Daftar fungsi yang akan diterapkan
feature_functions = [
    ('current_ratio', current_ratio),
    ('quick_ratio', quick_ratio),
    ('cash_ratio', cash_ratio),
    ('working_capital', working_capital),
    ('accounts_receivable_turnover', accounts_receivable_turnover),
    ('days_sales_outstanding', days_sales_outstanding),
    ('inventory_turnover', inventory_turnover),
    ('days_inventory_outstanding', days_inventory_outstanding),
    ('accounts_payable_turnover', accounts_payable_turnover),
    ('days_payable_outstanding', days_payable_outstanding),
    ('cash_conversion_cycle', cash_conversion_cycle),
    ('gross_profit_margin', gross_profit_margin),
    ('operating_profit_margin', operating_profit_margin),
    ('net_profit_margin', net_profit_margin),
    ('return_on_assets', return_on_assets),
    ('return_on_equity', return_on_equity),
    ('debt_to_equity_ratio', debt_to_equity_ratio),
    ('debt_to_assets_ratio', debt_to_assets_ratio),
    ('interest_coverage_ratio', interest_coverage_ratio),
    ('operating_expense_ratio', operating_expense_ratio),
    ('rd_to_revenue_ratio', rd_to_revenue_ratio),
    ('sga_to_revenue_ratio', sga_to_revenue_ratio),
    ('fixed_asset_turnover', fixed_asset_turnover),
    ('total_asset_turnover', total_asset_turnover),
    ('capital_expenditure_ratio', capital_expenditure_ratio),
    ('compensation_efficiency', compensation_efficiency),
    ('warranty_reserve_ratio', warranty_reserve_ratio),
    ('depreciation_rate', depreciation_rate),
    ('operating_cash_flow_to_net_income_ratio', operating_cash_flow_to_net_income_ratio),
    ('effective_tax_rate', effective_tax_rate),
    ('revenue_growth_rate', revenue_growth_rate),
    ('net_income_growth_rate', net_income_growth_rate),
    ('asset_growth_rate', asset_growth_rate),
    ('accrual_ratio', accrual_ratio),
    ('cash_flow_to_revenue_ratio', cash_flow_to_revenue_ratio),
    ('return_on_invested_capital', return_on_invested_capital),
    ('cash_return_on_capital_invested', cash_return_on_capital_invested),
    ('fixed_assets_to_long_term_debt_ratio', fixed_assets_to_long_term_debt_ratio),
    ('non_current_asset_turnover', non_current_asset_turnover),
    ('quarterly_gross_profit_stability', quarterly_gross_profit_stability),
    ('revenue_to_expense_growth_differential', revenue_to_expense_growth_differential),
    ('quarterly_cash_flow_quality', quarterly_cash_flow_quality),
    ('dividend_payout_ratio', dividend_payout_ratio),
    ('stock_based_compensation_to_operating_expense_ratio', stock_based_compensation_to_operating_expense_ratio),
    ('financial_leverage_index', financial_leverage_index),
    ('asset_coverage_ratio', asset_coverage_ratio),
    ('quarterly_margin_expansion', quarterly_margin_expansion),
    ('asset_utilization_ratio', asset_utilization_ratio),
    ('capacity_utilization_proxy', capacity_utilization_proxy),
    ('altman_z_score', altman_z_score),
    ('dupont_analysis_roe', dupont_analysis_roe),
    ('economic_value_added', economic_value_added),
    ('rd_efficiency_ratio', rd_efficiency_ratio),
    ('innovation_investment_ratio', innovation_investment_ratio),
    ('revenue_momentum', revenue_momentum),
    ('earnings_momentum', earnings_momentum),
    ('quarterly_earnings_quality_index', quarterly_earnings_quality_index),
    ('non_operating_items_ratio', non_operating_items_ratio)
]

# Fungsi yang memerlukan parameter kolom
column_based_functions = [
    ('revenues_qoq_growth', 'Revenues', qoq_growth),
    ('revenues_yoy_growth', 'Revenues', yoy_quarterly_growth),
    ('revenues_ttm', 'Revenues', trailing_twelve_months),
    ('revenues_acceleration', 'Revenues', quarterly_acceleration),
    ('revenues_seasonal_index', 'Revenues', seasonal_index),
    ('revenues_moving_avg', 'Revenues', moving_average_4q),
    ('revenues_run_rate', 'Revenues', quarter_run_rate),
    ('revenues_volatility', 'Revenues', quarterly_volatility),
    ('revenues_seasonal_dependency', 'Revenues', seasonal_dependency_index),
    ('net_income_qoq_growth', 'NetIncomeLoss', qoq_growth),
    ('net_income_yoy_growth', 'NetIncomeLoss', yoy_quarterly_growth),
    ('net_income_ttm', 'NetIncomeLoss', trailing_twelve_months),
    ('net_income_acceleration', 'NetIncomeLoss', quarterly_acceleration),
    ('net_income_seasonal_index', 'NetIncomeLoss', seasonal_index),
    ('net_income_moving_avg', 'NetIncomeLoss', moving_average_4q),
    ('net_income_run_rate', 'NetIncomeLoss', quarter_run_rate),
    ('net_income_volatility', 'NetIncomeLoss', quarterly_volatility),
    ('assets_qoq_growth', 'Assets', qoq_growth),
    ('assets_yoy_growth', 'Assets', yoy_quarterly_growth),
    ('assets_ttm', 'Assets', trailing_twelve_months),
    ('operating_income_qoq_growth', 'OperatingIncomeLoss', qoq_growth),
    ('operating_income_yoy_growth', 'OperatingIncomeLoss', yoy_quarterly_growth),
    ('operating_income_ttm', 'OperatingIncomeLoss', trailing_twelve_months),
    ('cash_qoq_growth', 'CashAndCashEquivalentsAtCarryingValue', qoq_growth),
    ('cash_yoy_growth', 'CashAndCashEquivalentsAtCarryingValue', yoy_quarterly_growth),
    ('equity_qoq_growth', 'StockholdersEquity', qoq_growth),
    ('equity_yoy_growth', 'StockholdersEquity', yoy_quarterly_growth),
    ('liabilities_qoq_growth', 'Liabilities', qoq_growth),
    ('liabilities_yoy_growth', 'Liabilities', yoy_quarterly_growth)
]

# Terapkan fungsi-fungsi feature engineering
print('\nMenerapkan feature engineering functions...')

# Fungsi dasar
for name, func in feature_functions:
    try:
        enhanced_df[name] = func(enhanced_df)
        print(f'✓ {name}')
    except Exception as e:
        print(f'✗ {name}: {str(e)}')

# Fungsi berbasis kolom
for name, column, func in column_based_functions:
    try:
        if column in enhanced_df.columns:
            enhanced_df[name] = func(enhanced_df, column)
            print(f'✓ {name}')
        else:
            print(f'✗ {name}: Column {column} not found')
    except Exception as e:
        print(f'✗ {name}: {str(e)}')

# Fungsi khusus
try:
    enhanced_df['quarterly_operating_leverage'] = quarterly_operating_leverage(enhanced_df)
    print('✓ quarterly_operating_leverage')
except Exception as e:
    print(f'✗ quarterly_operating_leverage: {str(e)}')

try:
    enhanced_df['quarterly_cash_burn_rate'] = quarterly_cash_burn_rate(enhanced_df)
    print('✓ quarterly_cash_burn_rate')
except Exception as e:
    print(f'✗ quarterly_cash_burn_rate: {str(e)}')

# Fungsi YTD untuk beberapa kolom utama
ytd_columns = ['Revenues', 'NetIncomeLoss', 'OperatingIncomeLoss']
for col in ytd_columns:
    try:
        if col in enhanced_df.columns:
            enhanced_df[f'{col.lower()}_ytd'] = ytd_performance(enhanced_df, col)
            print(f'✓ {col.lower()}_ytd')
    except Exception as e:
        print(f'✗ {col.lower()}_ytd: {str(e)}')

# Fungsi CAGR
try:
    enhanced_df['long_term_revenue_cagr'] = long_term_revenue_cagr(enhanced_df)
    print('✓ long_term_revenue_cagr')
except Exception as e:
    print(f'✗ long_term_revenue_cagr: {str(e)}')

# Fungsi trend
try:
    enhanced_df['operating_margin_trend'] = operating_margin_trend(enhanced_df)
    print('✓ operating_margin_trend')
except Exception as e:
    print(f'✗ operating_margin_trend: {str(e)}')

# Replace infinite values dengan NaN
enhanced_df = enhanced_df.replace([np.inf, -np.inf], np.nan)

print(f'\nEnhanced DataFrame shape: {enhanced_df.shape}')
print(f'Jumlah fitur asli: {len(complete_columns)}')
print(f'Jumlah fitur baru: {enhanced_df.shape[1] - len(complete_columns)}')
print(f'Total fitur: {enhanced_df.shape[1]}')

# Tampilkan statistik
print('\nStatistik Enhanced DataFrame:')
print(f'Data completeness: {(enhanced_df.count().sum() / (len(enhanced_df) * len(enhanced_df.columns)) * 100):.2f}%')

# Tampilkan sample data
print('\nSample enhanced data (first 5 rows, last 10 columns):')
print(enhanced_df.iloc[:5, -10:])

# Simpan enhanced dataframe
enhanced_filename = '/root/vynixmodelling/dataset/data_fundamental/TSLA_enhanced_features.csv'
enhanced_df.to_csv(enhanced_filename)
print(f'\nEnhanced DataFrame saved to: {enhanced_filename}')


Kolom dengan data lengkap (>= 70.0%): 57
Kolom: ['AccountsPayableCurrent', 'AccountsReceivableNetCurrent', 'AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment', 'AccumulatedOtherComprehensiveIncomeLossNetOfTax', 'AdditionalPaidInCapitalCommonStock', 'AllocatedShareBasedCompensationExpense', 'Assets', 'AssetsCurrent', 'CashAndCashEquivalentsAtCarryingValue', 'CommonStockValue']...

Filtered DataFrame shape: (57, 57)
Data completeness setelah filtering: 100.00%

Menerapkan feature engineering functions...
✓ current_ratio
✓ quick_ratio
✓ cash_ratio
✓ working_capital
✓ accounts_receivable_turnover
✓ days_sales_outstanding
✓ inventory_turnover
✓ days_inventory_outstanding
✓ accounts_payable_turnover
✓ days_payable_outstanding
✓ cash_conversion_cycle
✓ gross_profit_margin
✓ operating_profit_margin
✓ net_profit_margin
✓ return_on_assets
✓ return_on_equity
✓ debt_to_equity_ratio
✓ debt_to_assets_ratio
✓ interest_coverage_ratio
✓ operating_expense_ratio
✓ rd_to_revenue_rat

In [20]:
len(enhanced_df.columns)

150

In [21]:
len(enhanced_df)

57

In [22]:
# from eda import print_dataframe
# print_dataframe(pivoted_df)

In [23]:
# Filter DataFrame untuk periode 2012-Q2 hingga 2025-Q2
filtered_df = enhanced_df.loc["2012-Q2":"2025-Q2"]

# Cek kolom yang tidak memiliki nilai NaN di semua baris
complete_columns = filtered_df.columns[filtered_df.notna().all()].tolist()

# Cetak nama kolom dan total kolom
print(f"Kolom dengan data lengkap dari 2012-Q2 hingga 2025-Q2:")
for col in complete_columns:
    print(f"- {col}")

print(f"\nTotal kolom dengan data lengkap: {len(complete_columns)}")

Kolom dengan data lengkap dari 2012-Q2 hingga 2025-Q2:
- AccountsPayableCurrent
- AccountsReceivableNetCurrent
- AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment
- AccumulatedOtherComprehensiveIncomeLossNetOfTax
- AdditionalPaidInCapitalCommonStock
- AllocatedShareBasedCompensationExpense
- Assets
- AssetsCurrent
- CashAndCashEquivalentsAtCarryingValue
- CommonStockValue
- ComprehensiveIncomeNetOfTax
- CostOfRevenue
- EmployeeRelatedLiabilitiesCurrent
- GrossProfit
- IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest
- IncomeTaxExpenseBenefit
- IncreaseDecreaseInAccountsPayableAndAccruedLiabilities
- IncreaseDecreaseInAccountsReceivable
- IncreaseDecreaseInOtherNoncurrentLiabilities
- IncreaseDecreaseInPrepaidDeferredExpenseAndOtherAssets
- InterestCostsCapitalized
- InterestExpense
- InventoryNet
- InventoryWriteDown
- InvestmentIncomeInterest
- Liabilities
- LiabilitiesAndStockholdersEquity
- LiabilitiesCurrent
- NetCas

In [24]:
print_dataframe(filtered_df[complete_columns][["Revenues"]])

Revenues
Row 2012-Q2: 107201000.0
Row 2012-Q3: 164867000.0
Row 2012-Q4: 116744000.0
Row 2013-Q1: 30167000.0
Row 2013-Q2: 56820000.0
Row 2013-Q3: 106924000.0
Row 2013-Q4: 204242000.0
Row 2014-Q1: 561792000.0
Row 2014-Q2: 966931000.0
Row 2014-Q3: 1398277000.0
Row 2014-Q4: 413256000.0
Row 2015-Q1: 620542000.0
Row 2015-Q2: 1389891000.0
Row 2015-Q3: 2241695000.0
Row 2015-Q4: 2013496000.0
Row 2016-Q1: 939880000.0
Row 2016-Q2: 1894856000.0
Row 2016-Q3: 2831645000.0
Row 2016-Q4: 3198356000.0
Row 2017-Q1: 1147048000.0
Row 2017-Q2: 2417065000.0
Row 2017-Q3: 4715501000.0
Row 2017-Q4: 4046025000.0
Row 2018-Q1: 2696270000.0
Row 2018-Q2: 5485827000.0
Row 2018-Q3: 8470502000.0
Row 2018-Q4: 7000132000.0
Row 2019-Q1: 3408751000.0
Row 2019-Q2: 7410982000.0
Row 2019-Q3: 14235000000.0
Row 2019-Q4: 11759000000.0
Row 2020-Q1: 4541000000.0
Row 2020-Q2: 10891000000.0
Row 2020-Q3: 17194000000.0
Row 2020-Q4: 21461000000.0
Row 2021-Q1: 5985000000.0
Row 2021-Q2: 12021000000.0
Row 2021-Q3: 20792000000.0
Row 2021-Q

In [27]:
print(filtered_df.columns)


Index(['AccountsPayableCurrent', 'AccountsReceivableNetCurrent',
       'AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment',
       'AccumulatedOtherComprehensiveIncomeLossNetOfTax',
       'AdditionalPaidInCapitalCommonStock',
       'AllocatedShareBasedCompensationExpense', 'Assets', 'AssetsCurrent',
       'CashAndCashEquivalentsAtCarryingValue', 'CommonStockValue',
       ...
       'equity_yoy_growth', 'liabilities_qoq_growth', 'liabilities_yoy_growth',
       'quarterly_operating_leverage', 'quarterly_cash_burn_rate',
       'revenues_ytd', 'netincomeloss_ytd', 'operatingincomeloss_ytd',
       'long_term_revenue_cagr', 'operating_margin_trend'],
      dtype='object', name='metric', length=150)


In [28]:
filtered_df.to_csv('/root/vynixmodelling/dataset/data_fundamental/filtered_data.csv', index=False)


In [29]:
tsla_technical_data = pd.read_csv("/root/vynixmodelling/dataset/TSLA_from_2012.csv")

In [30]:
tsla_technical_data.head(5)

,time,open,high,low,close,Volume,Histogram,MACD,Signal,K,D,Turnover (Cr),10 MA Turnover,Turnover / 10MA (X),date
0,1325601000,1.929331,1.966665,1.843331,1.871998,13920793.0,-0.003331,-0.055094,-0.051764,79.322527,78.177718,2.605970,2.309154,1.128539,2012-01-03 14:30:00
1,1325687400,1.880665,1.911331,1.833332,1.847332,9450549.0,-0.003472,-0.056104,-0.052632,62.576316,74.606022,1.745830,2.209858,0.790019,2012-01-04 14:30:00
2,1325773800,1.850664,1.861997,1.789999,1.807998,15081495.0,-0.005409,-0.059393,-0.053984,45.004492,62.301112,2.726731,2.246949,1.213526,2012-01-05 14:30:00
3,1325860200,1.813332,1.852664,1.760665,1.793998,14794319.0,-0.006741,-0.062410,-0.055669,25.335543,44.305451,2.654098,2.042068,1.299711,2012-01-06 14:30:00
4,1326119400,1.799998,1.832666,1.741332,1.816665,13454278.0,-0.005268,-0.062254,-0.056986,19.420103,29.920046,2.444192,2.006065,1.218401,2012-01-09 14:30:00


In [32]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load data teknikal TSLA
tsla_technical_data = pd.read_csv('/root/vynixmodelling/dataset/TSLA_from_2012.csv')
tsla_technical_data['date'] = pd.to_datetime(tsla_technical_data['date'])
tsla_technical_data = tsla_technical_data.set_index('date')

# Filter data teknikal mulai dari Q2 2012 (1 April 2012)
tsla_technical_data = tsla_technical_data[tsla_technical_data.index >= '2012-04-01']
print(f'Technical data shape (from Q2 2012): {tsla_technical_data.shape}')
print(f'Technical data date range: {tsla_technical_data.index.min()} to {tsla_technical_data.index.max()}')

# Buat copy untuk mixed_df
mixed_df = tsla_technical_data.copy()

# Fungsi untuk mengkonversi quarter string ke date range
def get_quarter_date_range(quarter_str):
    """
    Mengkonversi format quarter (misal: '2012-Q2') ke date range
    """
    year = int(quarter_str.split('-')[0])
    quarter = quarter_str.split('-')[1]
    
    if quarter == 'Q1':
        start_date = pd.Timestamp(year=year, month=1, day=1)
        end_date = pd.Timestamp(year=year, month=3, day=31)
    elif quarter == 'Q2':
        start_date = pd.Timestamp(year=year, month=4, day=1)
        end_date = pd.Timestamp(year=year, month=6, day=30)
    elif quarter == 'Q3':
        start_date = pd.Timestamp(year=year, month=7, day=1)
        end_date = pd.Timestamp(year=year, month=9, day=30)
    else:  # Q4
        start_date = pd.Timestamp(year=year, month=10, day=1)
        end_date = pd.Timestamp(year=year, month=12, day=31)
    
    return start_date, end_date

# Tambahkan kolom fundamental ke mixed_df
print('\nMenambahkan data fundamental ke data teknikal...')

# Iterasi setiap quarter di filtered_df
for quarter_idx in filtered_df.index:
    start_date, end_date = get_quarter_date_range(quarter_idx)
    
    # Filter tanggal yang ada di mixed_df untuk quarter ini
    quarter_mask = (mixed_df.index >= start_date) & (mixed_df.index <= end_date)
    quarter_dates = mixed_df.index[quarter_mask]
    
    if len(quarter_dates) > 0:
        print(f'Processing {quarter_idx}: {len(quarter_dates)} trading days')
        
        # Tambahkan setiap kolom fundamental untuk quarter ini
        for col in filtered_df.columns:
            fundamental_value = filtered_df.loc[quarter_idx, col]
            
            # Isi nilai fundamental untuk semua hari trading dalam quarter ini
            mixed_df.loc[quarter_dates, col] = fundamental_value
    else:
        print(f'No trading days found for {quarter_idx}')

print(f'\nMixed DataFrame shape: {mixed_df.shape}')
print(f'Original technical columns: {len(tsla_technical_data.columns)}')
print(f'Added fundamental columns: {len(filtered_df.columns)}')
print(f'Total columns in mixed_df: {len(mixed_df.columns)}')

# Tampilkan sample data
print('\nSample mixed data (first 5 rows, last 5 columns):') 
print(mixed_df.iloc[:5, -5:])

# Cek data completeness
print('\nData completeness check:')
print(f'Technical data completeness: {(tsla_technical_data.count().sum() / (len(tsla_technical_data) * len(tsla_technical_data.columns)) * 100):.2f}%')
print(f'Mixed data completeness: {(mixed_df.count().sum() / (len(mixed_df) * len(mixed_df.columns)) * 100):.2f}%')

# Simpan mixed_df
mixed_filename = '/root/vynixmodelling/dataset/mixed_df.csv'
mixed_df.to_csv(mixed_filename)
print(f'\nMixed DataFrame saved to: {mixed_filename}')

# Tampilkan informasi tambahan
print('\nInformasi tambahan:')
print(f'Date range mixed_df: {mixed_df.index.min()} to {mixed_df.index.max()}')
print(f'Total trading days: {len(mixed_df)}')
print(f'Fundamental features added: {list(filtered_df.columns[:10])}...')  # Show first 10 fundamental columns

Technical data shape (from Q2 2012): (3372, 14)
Technical data date range: 2012-04-02 13:30:00 to 2025-08-28 13:30:00

Menambahkan data fundamental ke data teknikal...
Processing 2012-Q2: 63 trading days
Processing 2012-Q3: 63 trading days
Processing 2012-Q4: 61 trading days
Processing 2013-Q1: 60 trading days
Processing 2013-Q2: 64 trading days
Processing 2013-Q3: 63 trading days
Processing 2013-Q4: 63 trading days
Processing 2014-Q1: 60 trading days
Processing 2014-Q2: 62 trading days
Processing 2014-Q3: 63 trading days
Processing 2014-Q4: 63 trading days
Processing 2015-Q1: 60 trading days
Processing 2015-Q2: 62 trading days
Processing 2015-Q3: 63 trading days
Processing 2015-Q4: 63 trading days
Processing 2016-Q1: 60 trading days
Processing 2016-Q2: 63 trading days
Processing 2016-Q3: 63 trading days
Processing 2016-Q4: 63 trading days
Processing 2017-Q1: 61 trading days
Processing 2017-Q2: 62 trading days
Processing 2017-Q3: 63 trading days
Processing 2017-Q4: 63 trading days
Proc